# validate_paper_metrics — MobileNetV2 Nano FP32 (paper replica)

Derived from `code/train/mobilenet/validate_metrics.ipynb`. The DFire-mini, Clouds and dehazing parts and the Brevitas model were removed; only the paper test set and the FP32 model are kept.

**Paper result to reproduce:** MobileNetV2 Nano FP32, Table 5 (F1-Macro 95.93 %) and Table 8 (FP32 row).

**Historical run:** `~/uav/code/classifier_my_mobilenetv2/experiments/test_v04_mini_resnet_70k_full_ds`, checkpoint `weights/MY_MBLNET_V2_classifier__best_mean_F1.pt`. It is the epoch-86 checkpoint: same `epoch` field and identical tensors to `..._epoch=86.pt` (VERIFIED 2026-10-03).

**Weights used:** the local byte-identical copy `weights/MY_MBLNET_V2_classifier__best_mean_F1.pt` (sha256 `c8385799…`, checked in the Model cell).

**Kernel:** conda env `pytorch_23` (Python 3.10.13, the kernel of the original FP32 `train.ipynb`).

**Working directory:** this folder (`results_audit/mobilenet_paper_replica`).

Two evaluations:
- **A. Training-identical protocol.** `modules.val_epoch.eval_fn` with `get_val_loader()` (batch 64, `drop_last=True`, i.e. 24,320 of the 24,371 test images), exactly as in the training notebook. It must reproduce the logged epoch-86 values.
- **B. Independent per-image evaluation.** Full test set (`drop_last=False`, 24,371 images); TP/FP/TN/FN computed with numpy; predictions saved to `results/`. Restricted to the first 24,320 images, it must match A.

In [ ]:
# Safety: block any write to ~/uav or code/ (results_audit/common/write_guard.py)
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from common import write_guard
write_guard.install()
sys.dont_write_bytecode = True

In [ ]:
import json, hashlib, datetime, subprocess, importlib.metadata as md
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

import config
from modules.dataloaders import get_val_loader
from modules.model_mobilenetv2_mini_Resnet import MobileNetV2_MINI_RESNET
from modules.utils import load_checkpoint
from modules.loss import BCE_LOSS
import modules.val_epoch as val_epoch

DEVICE = config.DEVICE
print(f'Device is: {DEVICE}')
os.makedirs('results', exist_ok=True)

## Configuration check (values from the `test_v04` log header)

In [ ]:
expected_cfg = dict(NUM_CLASSES=2, CLASSES=['smoke', 'fire'], IMG_H=224, IMG_W=224, BATCH_SIZE=64,
                    LOSS_FN='BCE', SMOKE_PRECISION_WEIGHT=0.8, DS_LEN=None, BREVITAS_MODEL=False)
for k, v in expected_cfg.items():
    assert getattr(config, k) == v, (k, getattr(config, k), v)
print('config OK')
print('Checkpoint:', config.LOAD_MODEL_FILE)

# Dataloaders

In [ ]:
print("================================= TEST LOADER (paper protocol, drop_last=True) =================================")
test_loader_paper = get_val_loader(return_img_filename=False, drop_last=True)
print("================================= TEST LOADER (full, drop_last=False, with filenames) =================================")
test_loader_full = get_val_loader(return_img_filename=True, drop_last=False)
n_full = len(test_loader_full.dataset)
n_paper = len(test_loader_paper) * config.BATCH_SIZE
print(f'Test images: full = {n_full}, evaluated with drop_last = {n_paper}')
assert n_full == 24371, n_full   # Table 1: 4306 + 4181 + 15884

# Model

In [ ]:
model = MobileNetV2_MINI_RESNET().to(DEVICE)
epoch = load_checkpoint(config.LOAD_MODEL_FILE, model, device=DEVICE)   # load_state_dict(strict=True)
model.eval()
n_params = sum(p.numel() for p in model.parameters())
ckpt_sha256 = hashlib.sha256(open(config.LOAD_MODEL_FILE, 'rb').read()).hexdigest()
print(f'Epoch: {epoch} | Parameters: {n_params} | sha256: {ckpt_sha256}')
assert ckpt_sha256 == config.MODEL_FILE_SHA256, 'local weights differ from the paper checkpoint'
assert epoch == 86 and n_params == 68882

# A. Training-identical protocol

Expected (log `test_v04`, epoch 86 "VAL Stats" and the final "Testing with FULL TEST LOADER" line):
- smoke: Accuracy 0.954605, Precision 0.956552, Recall 0.946707, F1 0.951604
- fire: Accuracy 0.978248, Precision 0.952393, Recall 0.981989, F1 0.966964
- loss: total 11.07, smoke 7.07, fire 3.99

In [ ]:
expected_A = {'Accuracy': [0.9546052813529968, 0.9782483577728271],
              'Precision': [0.9565523862838745, 0.9523926973342896],
              'Recall': [0.9467073678970337, 0.9819888472557068],
              'F1': [0.951604425907135, 0.9669643640518188]}
loss_fn = BCE_LOSS(device=DEVICE, smoke_precision_weight=config.SMOKE_PRECISION_WEIGHT)
with torch.no_grad():
    losses_A, metrics_A = val_epoch.eval_fn(test_loader_paper, model, loss_fn, DEVICE)
diff_A = {k: [abs(a - b) for a, b in zip(metrics_A[k], expected_A[k])] for k in expected_A}
max_diff_A = max(max(v) for v in diff_A.values())
print('losses:', losses_A)
print('metrics:', metrics_A)
print(f'max |diff| vs log = {max_diff_A:.2e}')

# B. Independent per-image evaluation (full test set)

In [ ]:
rows = []
with torch.no_grad():
    for x, y, files in tqdm(test_loader_full, desc='Predicting'):
        p = torch.sigmoid(model(x.to(DEVICE))).cpu().numpy()
        y = y.numpy()
        for i in range(len(files)):
            rows.append({'file': files[i], 'y_smoke': int(y[i, 0]), 'y_fire': int(y[i, 1]),
                         'p_smoke': float(p[i, 0]), 'p_fire': float(p[i, 1])})
pred = pd.DataFrame(rows)
pred['file'] = pred['file'].str.replace(os.path.expanduser('~/uav/datasets/'), '', regex=False)
assert len(pred) == n_full

In [ ]:
def get_metrics(y_pred, y_true):
    y_pred, y_true = np.asarray(y_pred), np.asarray(y_true)
    TP = int((y_pred * y_true).sum()); FP = int((y_pred * (1 - y_true)).sum())
    TN = int(((1 - y_pred) * (1 - y_true)).sum()); FN = int(((1 - y_pred) * y_true).sum())
    n = TP + FP + TN + FN
    precision = TP / (TP + FP); recall = TP / (TP + FN)
    return {'TP': TP, 'FP': FP, 'TN': TN, 'FN': FN, 'N': n,
            'Accuracy': (TP + TN) / n, 'Precision': precision, 'Recall': recall,
            'F1': 2 * precision * recall / (precision + recall)}

def evaluate(df, threshold=0.5):
    # same binarization as torchmetrics Multilabel* (preds > threshold)
    out = {c: get_metrics((df[f'p_{c}'] > threshold).astype(int), df[f'y_{c}']) for c in ('smoke', 'fire')}
    out['F1_macro'] = (out['smoke']['F1'] + out['fire']['F1']) / 2
    return out

metrics_B_full = evaluate(pred)
metrics_B_paper_subset = evaluate(pred.iloc[:n_paper])
diff_AB = max(abs(metrics_B_paper_subset[c][k] - metrics_A[k][i])
              for i, c in enumerate(('smoke', 'fire')) for k in ('Accuracy', 'Precision', 'Recall', 'F1'))
print(json.dumps({'full_test': metrics_B_full, 'paper_subset': metrics_B_paper_subset}, indent=1))
print(f'max |B(subset) - A| = {diff_AB:.2e}')

# Comparison with the paper

Paper Table 8, FP32 row (%):
- Smoke: P 95.66, R 94.67, F1 95.16
- Fire: P 95.24, R 98.20, F1 96.70
- F1-Macro 95.93 (also Table 5)

In [ ]:
paper = {'smoke': {'Precision': 95.66, 'Recall': 94.67, 'F1': 95.16},
         'fire': {'Precision': 95.24, 'Recall': 98.20, 'F1': 96.70}, 'F1_macro': 95.93}
table = []
for c in ('smoke', 'fire'):
    for k in ('Precision', 'Recall', 'F1'):
        table.append({'class': c, 'metric': k, 'paper_%': paper[c][k],
                      'replica_A_%': round(100 * metrics_A[k][0 if c == 'smoke' else 1], 2),
                      'replica_full_test_%': round(100 * metrics_B_full[c][k], 2)})
f1m_A = 100 * (metrics_A['F1'][0] + metrics_A['F1'][1]) / 2
table.append({'class': 'macro', 'metric': 'F1', 'paper_%': paper['F1_macro'],
              'replica_A_%': round(f1m_A, 2), 'replica_full_test_%': round(100 * metrics_B_full['F1_macro'], 2)})
comparison = pd.DataFrame(table)
comparison['A_matches_paper'] = comparison['paper_%'] == comparison['replica_A_%']
comparison

# Save results and provenance

In [ ]:
def _git(*a):
    return subprocess.run(['git', *a], capture_output=True, text=True).stdout.strip()
record = {
    'timestamp': datetime.datetime.now().astimezone().isoformat(timespec='seconds'),
    'paper_result': 'MobileNetV2 Nano FP32 - Tables 5 and 8',
    'historical_run': config.PAPER_RUN_DIR, 'checkpoint': config.LOAD_MODEL_FILE, 'checkpoint_original': config.ORIGINAL_MODEL_FILE,
    'checkpoint_sha256': ckpt_sha256, 'checkpoint_epoch': epoch, 'n_params': n_params,
    'n_test_full': n_full, 'n_test_paper_protocol': n_paper,
    'A_training_protocol': {'losses': losses_A, 'metrics': metrics_A, 'max_abs_diff_vs_log': max_diff_A},
    'B_full_test': metrics_B_full, 'B_paper_subset': metrics_B_paper_subset, 'max_abs_diff_B_subset_vs_A': diff_AB,
    'comparison_with_paper': comparison.to_dict(orient='records'),
    'env': {'python': sys.version.split()[0], 'executable': sys.executable,
            'packages': {p: md.version(p) for p in ('torch', 'torchvision', 'albumentations', 'torchmetrics', 'numpy', 'opencv-python')},
            'cuda': torch.version.cuda, 'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None},
    'git': {'commit': _git('rev-parse', 'HEAD'), 'dirty': bool(_git('status', '--porcelain'))},
}
with open('results/validate_paper_metrics__test_v04_epoch86.json', 'w') as f:
    json.dump(record, f, indent=2, default=str)
pred.to_csv('results/predictions__test_v04_epoch86__full_test.csv', index=False, float_format='%.6f')
print('Saved results/validate_paper_metrics__test_v04_epoch86.json and predictions CSV')